# Godot Performance Analysis
Reads `test.csv` from the Godot performance monitor and exports PNG charts.

In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path

# pip install kaleido  <- needed for PNG export
# pip install plotly pandas

df = pd.read_csv('test.csv')
OUT = Path('perf_charts')
OUT.mkdir(exist_ok=True)

SAMPLES = list(range(len(df)))  # x-axis: sample index

print(f'Loaded {len(df)} samples, {len(df.columns)} columns')
df.describe().round(4)

Loaded 55 samples, 59 columns


,time/fps,time/process,time/physics_process,time/navigation_process,memory/static,memory/static_max,memory/msg_buf_max,object/objects,object/resources,object/nodes,...,navigation_3d/active_maps,navigation_3d/regions,navigation_3d/agents,navigation_3d/links,navigation_3d/polygons,navigation_3d/edges,navigation_3d/edges_merged,navigation_3d/edges_connected,navigation_3d/edges_free,navigation_3d/obstacles
count,55.0000,55.0000,55.0000,55.0000,5.500000e+01,5.500000e+01,55.0,55.0000,55.0,55.0,...,55.0,55.0,55.0,55.0,55.0,55.0,55.0,55.0,55.0,55.0
mean,139.4909,0.0125,0.0007,0.0000,7.202541e+07,7.202735e+07,4096.0,1596.9818,53.0,32.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
std,23.9172,0.0110,0.0004,0.0000,2.236599e+02,4.843452e+02,0.0,0.1348,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
min,1.0000,0.0000,0.0000,0.0000,7.202382e+07,7.202382e+07,4096.0,1596.0000,53.0,32.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25%,137.0000,0.0076,0.0005,0.0000,7.202544e+07,7.202742e+07,4096.0,1597.0000,53.0,32.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
50%,150.0000,0.0078,0.0006,0.0000,7.202544e+07,7.202742e+07,4096.0,1597.0000,53.0,32.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
75%,151.0000,0.0109,0.0008,0.0000,7.202544e+07,7.202742e+07,4096.0,1597.0000,53.0,32.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
max,152.0000,0.0492,0.0022,0.0001,7.202578e+07,7.202742e+07,4096.0,1597.0000,53.0,32.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 1 — FPS

In [2]:
avg_fps = df['time/fps'].mean()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=SAMPLES, y=df['time/fps'],
    mode='lines', name='fps',
    line=dict(color='#3B82F6', width=1.5),
    fill='tozeroy', fillcolor='rgba(59,130,246,0.08)'
))
fig.add_hline(
    y=avg_fps, line_dash='dash', line_color='#F59E0B',
    annotation_text=f'avg {avg_fps:.1f}', annotation_position='top right'
)
fig.add_hline(y=60, line_dash='dot', line_color='#10B981',
              annotation_text='60 fps target', annotation_position='bottom right')
fig.update_layout(
    title='FPS over samples',
    xaxis_title='sample', yaxis_title='fps',
    template='plotly_white', width=1000, height=400
)
fig.write_image(OUT / '01_fps.png')
fig.show()

## 2 — Memory usage

In [3]:
# convert bytes -> MB
mem_cols = {
    'memory/static':   ('static mem',  '#8B5CF6'),
    'video/video_mem': ('video mem',   '#06B6D4'),
    'video/texture_mem': ('texture mem', '#10B981'),
    'video/buffer_mem':  ('buffer mem',  '#F97316'),
}

fig = go.Figure()
for col, (label, color) in mem_cols.items():
    if col in df.columns:
        fig.add_trace(go.Scatter(
            x=SAMPLES, y=df[col] / 1024 / 1024,
            mode='lines', name=label,
            line=dict(color=color, width=1.5)
        ))

fig.update_layout(
    title='Memory usage over samples',
    xaxis_title='sample', yaxis_title='MB',
    template='plotly_white', width=1000, height=400
)
fig.write_image(OUT / '02_memory.png')
fig.show()

## 3 — Draw calls & raster

In [4]:
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=('draw calls per frame', 'objects & primitives drawn'),
    vertical_spacing=0.12
)

fig.add_trace(go.Scatter(
    x=SAMPLES, y=df['raster/total_draw_calls'],
    mode='lines', name='draw calls',
    line=dict(color='#EF4444', width=1.5)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=SAMPLES, y=df['raster/total_objects_drawn'],
    mode='lines', name='objects drawn',
    line=dict(color='#F97316', width=1.5)
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=SAMPLES, y=df['raster/total_primitives_drawn'],
    mode='lines', name='primitives drawn',
    line=dict(color='#FB923C', width=1, dash='dot')
), row=2, col=1)

fig.update_layout(
    title='Raster stats over samples',
    template='plotly_white', width=1000, height=500
)
fig.update_xaxes(title_text='sample', row=2)
fig.write_image(OUT / '03_draw_calls.png')
fig.show()

## 4 — Process & physics times

In [5]:
fig = go.Figure()

time_cols = {
    'time/process':            ('_process',   '#3B82F6'),
    'time/physics_process':    ('_physics',   '#10B981'),
    'time/navigation_process': ('_navigation','#8B5CF6'),
}
for col, (label, color) in time_cols.items():
    if col in df.columns:
        fig.add_trace(go.Scatter(
            x=SAMPLES, y=df[col] * 1000,  # seconds -> ms
            mode='lines', name=label,
            line=dict(color=color, width=1.5)
        ))

# 16.67ms budget line
fig.add_hline(y=16.67, line_dash='dot', line_color='#EF4444',
              annotation_text='16.67ms (60fps budget)', annotation_position='top right')

fig.update_layout(
    title='Per-frame process times (ms)',
    xaxis_title='sample', yaxis_title='ms',
    template='plotly_white', width=1000, height=400
)
fig.write_image(OUT / '04_process_times.png')
fig.show()

## 5 — Scene objects & node leaks

In [6]:
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=('objects & nodes', 'orphan nodes (leak indicator)'),
    vertical_spacing=0.12
)

fig.add_trace(go.Scatter(
    x=SAMPLES, y=df['object/objects'],
    mode='lines', name='objects',
    line=dict(color='#8B5CF6', width=1.5)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=SAMPLES, y=df['object/nodes'],
    mode='lines', name='nodes',
    line=dict(color='#F59E0B', width=1.5)
), row=1, col=1)

# Orphan nodes — should stay near 0; growth = leak
fig.add_trace(go.Scatter(
    x=SAMPLES, y=df['object/orphan_nodes'],
    mode='lines+markers', name='orphan nodes',
    line=dict(color='#EF4444', width=2),
    marker=dict(size=4, color='#EF4444')
), row=2, col=1)

fig.update_layout(
    title='Scene graph stats',
    template='plotly_white', width=1000, height=500
)
fig.update_xaxes(title_text='sample', row=2)
fig.write_image(OUT / '05_nodes.png')
fig.show()

## 6 — Frame budget breakdown (avg)

In [7]:
avg_fps   = df['time/fps'].mean()
frame_ms  = 1000 / avg_fps
proc_ms   = df['time/process'].mean() * 1000
phys_ms   = df['time/physics_process'].mean() * 1000
nav_ms    = df['time/navigation_process'].mean() * 1000 if 'time/navigation_process' in df.columns else 0
other_ms  = max(0, frame_ms - proc_ms - phys_ms - nav_ms)

labels = ['_process', '_physics', '_navigation', 'other / gpu']
values = [proc_ms, phys_ms, nav_ms, other_ms]
colors = ['#3B82F6', '#10B981', '#8B5CF6', '#94A3B8']

fig = go.Figure()
for label, val, color in zip(labels, values, colors):
    fig.add_trace(go.Bar(
        x=[val], y=['avg frame'],
        orientation='h', name=label,
        marker_color=color,
        text=[f'{val:.2f}ms'], textposition='inside'
    ))

fig.update_layout(
    barmode='stack',
    title=f'Frame budget breakdown — avg frame {frame_ms:.2f}ms ({avg_fps:.1f} fps)',
    xaxis_title='ms', template='plotly_white',
    width=1000, height=220,
    legend=dict(orientation='h', yanchor='bottom', y=1.1)
)
fig.write_image(OUT / '06_frame_budget.png')
fig.show()

## 7 — FPS vs draw calls (scatter)

In [8]:
fig = px.scatter(
    df,
    x='raster/total_draw_calls',
    y='time/fps',
    trendline='ols',
    labels={
        'raster/total_draw_calls': 'draw calls',
        'time/fps': 'fps'
    },
    title='FPS vs draw calls — correlation check',
    template='plotly_white',
    width=700, height=450
)
fig.update_traces(marker=dict(color='#3B82F6', size=6, opacity=0.7), selector=dict(mode='markers'))
fig.write_image(OUT / '07_fps_vs_drawcalls.png')
fig.show()

## 8 — Rolling stats (smoothed FPS + 1-sigma band)

In [9]:
W = 7  # window size
roll_mean = df['time/fps'].rolling(W, center=True).mean()
roll_std  = df['time/fps'].rolling(W, center=True).std()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=SAMPLES, y=df['time/fps'],
    mode='lines', name='raw fps',
    line=dict(color='rgba(59,130,246,0.3)', width=1)
))
fig.add_trace(go.Scatter(
    x=SAMPLES + SAMPLES[::-1],
    y=list(roll_mean + roll_std) + list((roll_mean - roll_std)[::-1]),
    fill='toself', fillcolor='rgba(59,130,246,0.12)',
    line=dict(color='rgba(0,0,0,0)'), name='±1σ band', showlegend=True
))
fig.add_trace(go.Scatter(
    x=SAMPLES, y=roll_mean,
    mode='lines', name=f'rolling mean (w={W})',
    line=dict(color='#3B82F6', width=2)
))
fig.add_hline(y=60, line_dash='dot', line_color='#10B981',
              annotation_text='60 fps', annotation_position='bottom right')

fig.update_layout(
    title=f'FPS rolling mean ± std (window={W})',
    xaxis_title='sample', yaxis_title='fps',
    template='plotly_white', width=1000, height=400
)
fig.write_image(OUT / '08_fps_rolling.png')
fig.show()

## Export summary

In [10]:
print('Charts written to:', OUT.resolve())
for f in sorted(OUT.glob('*.png')):
    print(' ', f.name)

print()
print('--- quick stats ---')
for col in ['time/fps', 'raster/total_draw_calls', 'object/orphan_nodes']:
    if col in df.columns:
        s = df[col]
        print(f'{col:40s}  min={s.min():.2f}  avg={s.mean():.2f}  max={s.max():.2f}')

Charts written to: /home/mare/Programs/Fax/Diploma/godot-fluidsim-lbm/stats/perf_charts
  01_fps.png
  02_memory.png
  03_draw_calls.png
  04_process_times.png
  05_nodes.png
  06_frame_budget.png
  07_fps_vs_drawcalls.png
  08_fps_rolling.png

--- quick stats ---
time/fps                                  min=1.00  avg=139.49  max=152.00
raster/total_draw_calls                   min=2.00  avg=2.00  max=2.00
object/orphan_nodes                       min=0.00  avg=0.00  max=0.00
